# Exploring Generative LLMs

In this notebook, we will learn how to interact with open-source generative language models using [ollama](https://github.com/ollama/ollama-python). We will cover everything from installing dependencies to downloading a model (for example, [gemma3](https://ollama.com/library/gemma3:270m)), generating text with various parameters, and using the model to classify text in a pandas DataFrame.

**Why does this matter?** Communication science has long studied how messages are crafted and interpreted. Generative models "craft" text based on patterns learned from human language, so experimenting with them gives us insight into language construction, persuasion techniques, and narrative formation. This also sets up a nice context for us to start understanding machine learning.

**What we will learn:**
- How to set up our environment and download a model
- How to generate text using simple and tuned prompts
- How these technical concepts relate to real-world communication principles
- How to use open-source LLMs for a simple classification task on pandas data

**Before you start:** cells in this notebook depend on each other. `client` (created in Section 1) and `prompt` (created in Section 3) are reused several cells later, so run the cells in order from top to bottom. If you jump into the middle and get a `NameError`, run the earlier cells first.

## 1. Setting Up the Environment

Before we start, make sure you have [ollama](https://ollama.com/download) installed on your machine (follow the installation instructions for your operating system) and that the Ollama app/service is running in the background.

We also need the `ollama` Python package, which gives us a Python interface to talk to the local Ollama service.

In [1]:
# Install the Python client for ollama.
# If you run into issues, see: https://github.com/ollama/ollama-python/blob/main/README.md
# !pip install ollama

In [ ]:
# Troubleshooting only: some versions of the ollama package rely on docstring_parser internally.
# If the import cell below raises an error mentioning docstring_parser, run this cell then try again.
!pip install docstring_parser

In [2]:
import ollama

# Create a client that talks to the Ollama service running on your machine.
# By default this connects to http://localhost:11434, which is where Ollama listens once installed.
client = ollama.Client()

**Breaking down this cell:**

- **`import ollama`**: loads the Python package we just installed.
- **`ollama.Client()`**: creates a client object: our handle for sending requests to (and getting responses from) the Ollama service running locally on your computer.
- **`client = ...`**: we save that handle in a variable called `client`, which we will reuse in every section below to interact with a model. This is why cells need to run in order: if `client` doesn't exist yet, later cells that use it will fail with a `NameError`.

## 2. Downloading the Model

Generative models need to be downloaded before they can be used.
We'll use `gemma3:270m` as our example model. The `270m` means it has 270 million parameters. A list of other available models can be found [here](https://ollama.com/search).

In [3]:
# Download the gemma3:270m model. This only needs to be run once per machine.
# It needs an internet connection and some disk space, and may take a minute or two.
!ollama pull gemma3:270m

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling 735af2139dc6: 100% ▕██████████████████▏ 291 MB                         
pulling 4b19ac7dd2fb: 100% ▕██████████████████▏  476 B                         
pulling 3e2c24001f9e: 100% ▕██████████████████▏ 8.4 KB                         
pulling 339e884a40f6: 100% ▕██████████████████▏   61 B                         
pulling 74156d92caf6: 100% ▕██████████████████▏  490 B                         
verifying sha256 digest 
writing manifest 
success 


**Breaking down this cell:**

- The `!` at the start runs a shell/terminal command directly from the notebook (not Python code).
- **`ollama pull gemma3:270m`**: tells Ollama to download the model's weights to your machine, the same way `git pull` downloads code, but here it's downloading the trained model itself. Once downloaded, the model stays on your machine and doesn't need to be pulled again.

**What you should see:** a progress bar for the download, ending in `success`.

## 3. Generating Text with the Model

Now that the model is downloaded and we have a `client` ready to talk to it, let's generate our first piece of text.

In [4]:
# The starting text we give the model to continue
prompt = "Once upon a time in a land far away,"

# Ask the model to continue the prompt, using the client we created in Section 1
result = client.chat(
    model="gemma3:270m",
    messages=[{"role": "user", "content": prompt}]
)

# The generated text is nested inside the response, so we pull it out here
print(result['message']['content'])

Once upon a time in a land far away,
A world of shimmering, emerald hues,
Where ancient magic flowed in the breeze,
And whispered secrets through the trees.



In [5]:
result

ChatResponse(model='gemma3:270m', created_at='2026-09-25T09:51:55.514241Z', done=True, done_reason='stop', total_duration=21827802084, load_duration=21451863167, prompt_eval_count=19, prompt_eval_duration=82949000, eval_count=38, eval_duration=290053000, message=Message(role='assistant', content='Once upon a time in a land far away,\nA world of shimmering, emerald hues,\nWhere ancient magic flowed in the breeze,\nAnd whispered secrets through the trees.\n', thinking=None, images=None, tool_name=None, tool_calls=None), logprobs=None)

**Breaking down this cell:**

- **`prompt = "..."`**: the text we want the model to continue. This is just a Python string, and we're saving it in a variable so we can reuse it later.
- **`client.chat(...)`**: sends a request to the model and waits for its response. This is the main function we'll use throughout this notebook.
- **`model="gemma3:270m"`**: tells Ollama which downloaded model to use.
- **`messages=[{"role": "user", "content": prompt}]`**: the conversation we're sending, formatted as a list of messages. Each message is a dictionary with a `"role"` (who is "speaking": here, `"user"`, meaning us) and `"content"` (what they said). This list-of-dictionaries format is how most chat-based LLM APIs expect input, even for a single message.
- **`result['message']['content']`**: `result` is a nested dictionary; this digs into it to pull out just the generated text, the same way you'd access any nested dictionary/list in Python.

**What you should see:** a short story continuing from the prompt. Since generation involves randomness, your exact output will differ each time you run the cell, and will differ from a classmate's too.

**Try it yourself:**
- Change `prompt` to something else (a question, a different opening line) and re-run the cell. How does the model's continuation change?

## 4. Experimenting with Generation Parameters

Generative models have parameters that control how text is generated.

**Key parameters:**
- **`num_predict`**: the maximum number of tokens (words, or parts of words) to generate.
- **`temperature`**: controls randomness. Lower values (e.g. 0.5) make the output more predictable; higher values (e.g. 1.0) increase creativity, but can make the text less coherent.
- **`top_k`**: at each generation step, only the top `k` most likely next tokens are considered. This narrows the model's "vocabulary" down to its most likely choices at each step.

In [6]:
# Generate text again with the same prompt, but with tuned parameters this time
response_tuned = client.chat(
    model="gemma3:270m",
    messages=[{"role": "user", "content": prompt}],
    options={
        "num_predict": 30,   # stop after about 30 tokens, so the output is short
        "temperature": 0.5,  # a balance between creativity and coherence
        "top_k": 50          # only consider the 50 most likely next tokens at each step
    }
)

print(response_tuned['message']['content'])

Once upon a time in a land far away,
Life was simple, a vibrant, sunlit scene.
The air was sweet with blossoms,


**Breaking down this cell:**

- **`options={...}`**: a dictionary of extra settings passed alongside the model and messages. This is where `num_predict`, `temperature`, and `top_k` go.
- Everything else is the same `client.chat()` call as Section 3, reusing the same `prompt`.

**What you should see:** a shorter, more constrained continuation than Section 3's default generation, since `num_predict=30` caps the length.

**Try it yourself:**
- Set `temperature` to `0.0` and run the cell a few times. Does the output stay the same or change each time? Now try `temperature=1.0`. What changes?
- Try a much smaller `top_k`, like `5`. Does the text feel more repetitive or predictable?

## 5. More ways to interact

Let's reflect on the process:

- **Narrative structure:** the prompt acts like an opening line in a story or a hook in a speech. How does the model build on it?
- **Message framing:** the parameters (`temperature`, `top_k`) determine the "tone" or style of the output, similar to how framing affects how a human-written message is perceived.

**Exercise:** try a new prompt below, and compare its tone to the fairy-tale prompt from Section 3.

In [12]:
# Try it yourself: write your own prompt and see how the model continues it
my_prompt = "The most important thing about social media is"

result_custom = client.chat(
    model="gemma3:270m",
    messages=[{"role": "user", "content": my_prompt}]
)

print(result_custom['message']['content'])

The most important thing about social media is **connection and community**.



**Try it yourself:** swap `my_prompt` for a persuasive, descriptive, or humorous prompt of your own. How do you think the parameters and prompt wording mirror techniques used by professional communicators?

## 6. Additional Exercises and Exploration

How might understanding these AI parameters help us analyze or even craft persuasive messages in advertising, politics, or social media?

In [13]:
# Try it yourself: run this media-focused prompt and compare its tone to the fairy-tale one from Section 3
media_prompt = "In today's rapidly changing media landscape, communication is more important than ever..."

result_media = client.chat(
    model="gemma3:270m",
    messages=[{"role": "user", "content": media_prompt}]
)

print(result_media['message']['content'])

That's a very insightful and common sentiment! It highlights the crucial importance of communication in today's fast-paced and interconnected world. 



**Try it yourself:** how does the model handle this modern, media-focused prompt compared to the fairy-tale opening from Section 3? Does it stay in a similar "voice," or does it adapt?

## 7. Using an LLM for Classification on a Pandas DataFrame

In this example, we'll see how to use the model for a simple classification task on a pandas DataFrame. We have a small DataFrame with a column `film_plot` containing film plot descriptions, and we'll ask the model to rate each plot's uniqueness on a scale of 1 (very generic) to 5 (extremely novel). This shows how LLMs can be used for labelling tasks on structured data, similar to the `.apply()` pattern we used earlier in the course.

In [15]:
import pandas as pd

# A small sample DataFrame with 5 film plots
df = pd.DataFrame({
    'film_plot': [
        "A group of friends embark on a quest to find a hidden treasure in the mountains.",
        "In a dystopian future, society is divided by class and a young rebel fights against an oppressive regime.",
        "A love story unfolds between two strangers who meet on a rainy day in a bustling city.",
        "A detective investigates a series of bizarre murders on Mars that seem to be connected to an ancient curse.",
        "An underdog sports team overcomes insurmountable odds to win the championship against all expectations."
    ]
})

def rate_uniqueness(plot):
    # Build a prompt that includes the plot text and asks for a single number back.
    # Being explicit ("Only output the rating number") helps keep the model's answer easy to parse.
    prompt = (
        f"Rate the uniqueness of the following film plot on a scale of 1 to 5, "
        f"where 1 is very generic and 5 is extremely novel:\n\n"
        f"{plot}\n\n"
        "Only output the rating number."
    )
    response = client.chat(
        model="gemma3:270m",
        messages=[{"role": "user", "content": prompt}],
        options={
            "temperature": 0.7  # a bit of variation, but still fairly consistent ratings
        }
    )
    # .strip() removes any extra whitespace/newlines around the model's answer
    rating = response['message']['content'].strip()
    return rating

# Apply the function to every row: this calls the model once per plot, so it takes a few seconds
df['uniqueness_rating'] = df['film_plot'].apply(rate_uniqueness)

df

,film_plot,uniqueness_rating
0,A group of friends embark on a quest to find a...,5
1,"In a dystopian future, society is divided by c...",5
2,A love story unfolds between two strangers who...,4
3,A detective investigates a series of bizarre m...,5
4,An underdog sports team overcomes insurmountab...,5


**Breaking down this cell:**

- **`rate_uniqueness(plot)`**: a function that takes one film plot (a string) and returns the model's rating for it, following the same `client.chat()` pattern from Sections 3 and 4.
- **The f-string prompt**: builds a custom instruction for each plot by inserting its text into a template. Ending with "Only output the rating number" is a prompting technique to keep the model's answer short and easy to use, rather than a full sentence explaining its reasoning.
- **`df['film_plot'].apply(rate_uniqueness)`**: calls `rate_uniqueness()` once per row, the same `.apply()` pattern used earlier in the course, except each "computation" here is a live call to the language model instead of a plain Python calculation.

**What you should see:** a new `uniqueness_rating` column, with the model's rating for each plot. Since the model is generating text, not guaranteed structured output, the values may come back as text strings (possibly with extra words) rather than clean numbers.

**Try it yourself:**
- Run `df['uniqueness_rating'].dtype`. Is it numeric, or still text (`object`)? If it's text, how would you use `pd.to_numeric()` (like we did earlier with the sentiment columns) to convert it, and what might go wrong if the model didn't follow instructions and returned something other than a plain number?
- Try changing the prompt's wording (e.g. asking for a one-sentence justification too) and see how that changes what comes back.

## 8. Conclusion

In this notebook, we covered:
1. How to set up an environment for working with LLMs in Python using **ollama**.
2. How to download a pre-trained model (`gemma3:270m`) using ollama's CLI.
3. How to generate text using simple and tuned prompts via the ollama chat API.
4. How generation parameters relate to communication techniques like message framing.
5. How to use an open-source LLM for a simple classification task on pandas data (rating film plot uniqueness).